In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MASTER CONFIGURATION — edit all pipeline parameters here
# ══════════════════════════════════════════════════════════════════════════════

# ── EBSD input ────────────────────────────────────────────────────────────────
CTF_FILE        = r'C:\Development\EBSD datasets\UKAEA__OFHCCu\OFHC_Cu_dataset\EBSD_pre\warp_out.ctf'
MIN_GRAIN_SIZE  = 10    # pixels — grains smaller than this are flagged -2
MISORI_TOL      = 8     # degrees — grain-boundary misorientation threshold
CROP_REGION     = [5, 5, 80, 80]   # [xstart%, ystart%, xend%, yend%]

# ── Simulated grain structure ─────────────────────────────────────────────────
INPUT_DASHBOARD = r'C:\Development\UPXO\upxo_library\src\upxo\demos\repGen\tgs2d.xls'
TSLICE_INDEX    = 10    # index into pxt.m[] used to pick the best-matching tslice

# ── Smoothing / tessellation ──────────────────────────────────────────────────
SMOOTH_ITER     = 10    # Taubin smoothing iterations
SMOOTH_LAMBDA   = 0.5
SMOOTH_MU       = -0.53
TRIM_BOUNDS     = (5, 5, 95, 95)  # (xmin%, ymin%, xmax%, ymax%) — % of LFI dims
AREA_THRESHOLD  = 1     # merge grains with area <= this value

# ── Meshing ───────────────────────────────────────────────────────────────────
MESH_SIZE_GB        = 0.75   # element size at grain boundaries (px)
MESH_SIZE_BULK      = 4.5    # element size in grain interiors (px)
MESH_ORDER          = 1      # 1 = linear, 2 = quadratic
MESH_ALGO           = 8      # 6=Frontal, 8=Frontal-Delaunay for quads
RECOMBINE_TO_QUADS  = True   # recombine triangles → quads (MESH_ALGO=8 only)

# ── Export ────────────────────────────────────────────────────────────────────
OUT_DIR  = r'C:\Development\UPXO\upxo_library\src\upxo\_written_data'
BASENAME = 'repgen_gs_mesh'

print("Configuration loaded.")

### Smoothing parameter guide

| Parameter | Default | Effect |
|---|---|---|
| `SMOOTH_ITER` | 10 | Number of Taubin two-pass iterations |
| `SMOOTH_LAMBDA` | 0.5 | Forward-pass vertex displacement weight |
| `SMOOTH_MU` | −0.53 | Backward-pass weight (must be negative) |
| `TRIM_BOUNDS` | (5, 5, 95, 95) | RVE clip as `(xmin%, ymin%, xmax%, ymax%)` of LFI size |
| `AREA_THRESHOLD` | 1 | Grains with ≤ this many pixels are merged before tessellation |

---

**`SMOOTH_ITER`** — Each iteration runs one forward (λ) pass then one backward (μ) pass. More iterations flatten jagged pixellated boundaries further but progressively erode thin features (twin lamellae, narrow grain necks). Beyond ~20–30 iterations, thin lamellae can collapse to zero-area slivers. Too few (<5) and the staircase pixel artefacts remain visible.

**`SMOOTH_LAMBDA`** — Forward-pass weight. Each vertex moves `λ × (neighbourhood_mean − vertex)`. Higher values flatten faster per iteration but amplify numerical noise in badly-shaped polygons. Must be in (0, 1); 0.3–0.6 is typical.

**`SMOOTH_MU`** — Backward-pass weight (must be negative, and `|μ| > λ`). The backward pass counteracts the shrinkage introduced by the forward pass. The passband condition `1/λ + 1/μ > 0` ensures the filter removes high-frequency jaggedness without inflating or deflating polygon area.
- If `|μ|` is only slightly larger than `λ` (e.g. λ=0.5, μ=−0.51) → very slow smoothing, almost no net displacement per iteration.
- If `|μ| >> λ` (e.g. μ=−0.9) → surface inflation and eventual polygon self-intersection.

**`TRIM_BOUNDS`** — RVE clip rectangle as percentages `(xmin%, ymin%, xmax%, ymax%)` of the LFI pixel dimensions, analogous to `CROP_REGION` for EBSD. Converted to pixel coordinates automatically inside `smooth_mc_slices` using each slice's `lgi.shape`. Too tight (large margins): genuine boundary grains are clipped to slivers and removed in Step 5. Too loose (margins near 0/100): tessellation boundary artefacts from unbounded Voronoi cells survive into the mesh. 5–10% margin on each side is typical.

**`AREA_THRESHOLD`** — Grains with ≤ this many pixels are merged into their largest neighbour before tessellation. At value 1, only isolated single-pixel islands are consumed — safe for typical MC grain structures. Raising to 3–5 removes numerical noise grains but risks merging genuinely thin twin lamella segments, permanently erasing them before smoothing begins.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy

import cc3d

from upxo.repgen.repgen2dmcgs import repgen2d
from upxo.interfaces.defdap.ebsd_reader import EBSDReader
from upxo.ggrowth.mcgs import mcgs
from upxo.pxtal.geometrification import GrainManifold2D
from upxo.gsdataops.gid_ops import find_neighs2d
from shapely.geometry import MultiPolygon

import upxo.viz.ebsdviz as ebsdviz
import upxo.gsdataops.grid_ops as gridOps
from upxo.viz import gsviz


Imports `repgen2d`, `EBSDReader`, and `ebsdviz`. `%matplotlib inline` keeps plots inline.


## 1. Load EBSD file


In [ ]:
rdr = EBSDReader.from_file(CTF_FILE, min_grain_size=MIN_GRAIN_SIZE, misori_tol=MISORI_TOL)
print(rdr)
print(f'\nRaw map shape  : {rdr.shape}')
print(f'Step size      : {rdr.step_size} µm')
print(f'Grains (raw)   : {rdr.n_grains}')


## 2. Crop to region of interest
`[xstart%, ystart%, xend%, yend%]` — percentages of the full map dimensions.


In [ ]:
rdr = rdr.crop(CROP_REGION, inplace=False)
print(rdr)


Crops the map to `[xstart%, ystart%, xend%, yend%]` as percentages of full dimensions. Returns a new `EBSDReader`.


## 3. Build `repgen2d` with EBSD target


In [ ]:
rg = repgen2d.from_tgs(tgs=None, tgstype='ebsd2d', ebsd_file=CTF_FILE)
print(f'iroute   : {rg.iroute}')
print(f'tgstype  : {rg.tgstype}')


`repgen2d.from_tgs(..., tgstype='ebsd2d')` creates the representative-generation object backed by an EBSD target.


## 4. Clean (characterise): fill non-indexed / boundary pixels, compute morphology & neighbours
Uses `EBSDReader.characterise()` — fills non-positive pixels, computes per-grain morphology and first-order neighbour graph.


In [ ]:
rg.clean_and_rechar_from_rdr(rdr, connectivity=4, min_grain_size=MIN_GRAIN_SIZE)


`characterise()` fills non-indexed pixels, labels grains, computes morphology, quaternions, and 4-connected neighbour graph. Results are stored in `rg.*_ebsd` slots.


## 5. Compute grain statistics


In [ ]:
rg.compute_ebsd_stats()


Per-property mean / std / min / max / count from `rg.stat_ebsd` (populated by `compute_ebsd_stats()`).


## 6. Grain size distribution


In [ ]:
fig, ax = rg.see_distr(prop="area", source="ebsd", nbins=40)
plt.show()


Grain area histogram (pixel counts × step² = µm²). Useful for assessing grain size bimodality or outliers.


## 7. Visualise cleaned grain map with boundaries


In [ ]:
fig, axes = rdr.plot_grain_structure_with_boundaries(
    show_euler_panel=True,
    euler_channel=0,          # 0=phi1, 1=Phi, 2=phi2
    boundary_color=(0, 0, 0),
    figsize=(16, 7),
    dpi=150,
    suptitle='Cleaned EBSD target — grain labels + phi1 + boundaries',
)


Side-by-side grain label map + phi1 Euler channel with grain boundaries overlaid. Uses `rdr.plot_grain_structure_with_boundaries()` — `rdr` already holds the cleaned arrays after `characterise()`.


## 8. All three Euler channels


In [ ]:
rdr.plot_euler_maps(
    show_boundaries=True,
    figsize=(15, 5),
    dpi=120,
    suptitle='Cleaned EBSD — Bunge Euler angle maps with grain boundaries',
)


All three Bunge Euler channels (φ₁, Φ, φ₂) plotted with boundary overlay for texture inspection.


## 9 MDF calcualtion
Compute grain-boundary MDF (24×24 cubic-symmetry search). Per-grain quaternions are averaged (positive-w hemisphere) before disorientation calculation. Returns the `mdf` dict needed for down the tream pipelines.


In [ ]:
import upxo.xtalphy.crystal_orientation as _co
from upxo.xtalphy.crystal_orientation import compute_mdf_from_quats

mdf = compute_mdf_from_quats(rg.lfi_ebsd, rg.quat_ebsd, rg.neigh_gid_ebsd)

print(f'Unique grain-boundary pairs : {mdf["n_pairs"]}')
print(f'Misorientation range        : {mdf["miso_deg"].min():.2f}° – {mdf["miso_deg"].max():.2f}°')
print(f'Mean disorientation         : {mdf["mean_angle"]:.2f}° ± {mdf["std_angle"]:.2f}°')


In [ ]:
import upxo.xtalphy.crystal_orientation as _co
import upxo.viz.ebsdviz as _ev

from upxo.xtalphy.crystal_orientation import detect_mdf_peaks
import upxo.viz.ebsdviz as ebsdviz

peaks = detect_mdf_peaks(mdf)

print(f"{'Detected peaks':─^55}")
for label, (csl_name, delta) in zip(peaks['peak_labels'], peaks['csl_nearest']):
    print(f'  {label}')

ebsdviz.plot_mdf(mdf, peaks)
plt.show()


Detects histogram peaks (`scipy.signal.find_peaks`), matches them to cubic CSL angles (Σ3 60°, Σ5 36.9°, …), fits a KDE, and plots the annotated MDF. Results stored in `peaks` dict.


## Selecrt the peaks of interest from MDF


In [ ]:
selected_peaks = ebsdviz.mdf_peak_selector(peaks)


## View the selected peaks and replot


In [ ]:
ebsdviz.plot_mdf_selected(mdf, peaks, selected_peaks)
plt.show()
print(f'Selected peak angles : {selected_peaks["angles"]}')


## `peaks` dictionary keys

| Key | Description |
|-----|-------------|
| `peak_labels` | Pak label names e.g. `"38.5° ← near Σ5 (Δ=+1.6°)"` |
| `peak_angles` | Detected peak positions in degrees |
| `peak_indices` | Histogram bin indices of detected peaks |
| `csl_nearest` | List of `(csl_name, delta_deg)` tuples — closest CSL to each peak |
| `csl` | Dict mapping CSL name → reference angle (e.g. `{'Σ3': 60.0, 'Σ5': 36.9, ...}`) |
| `csl_tol` | Tolerance in degrees for CSL matching (used by `segregate_csl_pairs`) |
| `kde_x` | KDE evaluation points (degrees) |
| `kde_y` | KDE density values |


## 11. Segregate grain pairs by CSL type


In [ ]:
import upxo.xtalphy.crystal_orientation as _co
import upxo.viz.ebsdviz as _ev

from upxo.xtalphy.crystal_orientation import segregate_csl_pairs
import upxo.viz.ebsdviz as ebsdviz

# Segregate using selected_peaks from the widget cell
csl_grains = segregate_csl_pairs(mdf, selected_peaks,
                csl=peaks['csl'], csl_tol=peaks['csl_tol'])

# Summary table
print(f"{'CSL type':<18} {'ref °':>6}  {'pairs':>6}  {'grains':>7}")
print('─' * 42)
for lbl, info in csl_grains.items():
    print(f"{lbl:<18} {info['csl_angle']:>6.2f}  "
          f"{info['n_pairs']:>6}  {info['n_grains']:>7}")


Uses `mdf['pairs']` + `mdf['miso_deg']` to group boundary pairs within `csl_tol` of each selected CSL reference angle. `csl_grains` maps each label to its pairs, grain IDs, and counts.


In [ ]:
csl_grains


In [ ]:
ebsdviz.plot_csl_grain_map(
    rg.lfi_ebsd,
    csl_grains,
    figsize=(11, 6),
    dpi=140,
    suptitle='CSL grain participation map — EBSD target',
)
plt.show()


Grain map coloured by CSL participation. Each CSL type gets a distinct tableau colour; grains touching two types are blended. Non-CSL grains are semi-transparent grey.


## 12. CSL twinned volume fractions


In [ ]:
import upxo.xtalphy.crystal_orientation as _co
from upxo.xtalphy.crystal_orientation import csl_volume_fractions

vf = csl_volume_fractions(rg.lfi_ebsd, csl_grains)

# ── Table ─────────────────────────────────────────────────────────────────────
print(f"{'CSL type':<18} {'ref °':>6}  {'grains':>7}  "
      f"{'pixels':>8}  {'VF (indexed)':>13}  {'VF (total)':>11}")
print('─' * 70)
for lbl, v in vf.items():
    print(f"{lbl:<18} {v['csl_angle']:>6.2f}  {v['n_grains']:>7}  "
          f"{v['n_pixels']:>8}  {v['vf_indexed']:>12.2%}  {v['vf_total']:>10.2%}")

# ── Bar chart ─────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

palette   = list(mcolors.TABLEAU_COLORS.values())
labels    = list(vf.keys())
vf_vals   = [vf[l]['vf_indexed'] * 100 for l in labels]
colors    = [palette[i % len(palette)] for i in range(len(labels))]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(labels, vf_vals, color=colors, edgecolor='k', linewidth=0.5)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_ylabel('Volume fraction of indexed area (%)')
ax.set_title('CSL twinned volume fractions — EBSD target')
ax.set_ylim(0, max(vf_vals) * 1.25)
plt.tight_layout()
plt.show()


Pixel count of grains in each CSL class divided by total indexed pixels. `vf_indexed` is the more meaningful metric; `vf_total` accounts for non-indexed regions. Note: fractions can exceed 100% total if grains participate in multiple CSL types.


## 13. Identify parent grains for each CSL type

For every CSL boundary pair, the **larger** grain (by pixel area) is designated the **parent** and the smaller the **twin/child**. A grain can appear as parent in some pairs and twin in others (twin chains: A→B→C). Grains are therefore classified as:

- **Pure parent** – larger grain in *every* pair it participates in  
- **Pure twin** – smaller grain in *every* pair  
- **Intermediate** – both roles across different pairs (secondary twinning)


In [ ]:
import upxo.xtalphy.crystal_orientation as _co
import upxo.viz.ebsdviz as ebsdviz
from upxo.xtalphy.crystal_orientation import identify_parent_grains
from upxo.viz.ebsdviz import print_parent_grain_summary, plot_parent_grain_summary

parent_info = identify_parent_grains(csl_grains, rg.prop_ebsd)

print_parent_grain_summary(parent_info, csl_grains)

fig, ax = plot_parent_grain_summary(
    parent_info,
    figsize=(8, 4),
    dpi=120,
    title='Parent / twin / intermediate grain counts per CSL type',
)


In [ ]:
import upxo.viz.ebsdviz as ebsdviz
from upxo.viz.ebsdviz import plot_parent_twin_map

fig, axes = plot_parent_twin_map(
    rg.lfi_ebsd,
    parent_info,
    figsize=(10, 5),
    dpi=130,
    alpha_bg=0.10,
    suptitle='Parent / twin / intermediate grain domains per CSL type',
)


In [ ]:
import upxo.viz.ebsdviz as ebsdviz
from upxo.viz.ebsdviz import plot_combined_parent_twin_map

fig, ax = plot_combined_parent_twin_map(
    rg.lfi_ebsd,
    parent_info,
    figsize=(10, 5),
    dpi=130,
    alpha_bg=0.08,
)


## 14. Property & topology statistics by grain role

Select which morphological / topological properties and grain-role groups to include in the analysis, then run the next cell.


In [ ]:
import upxo.viz.ebsdviz as ebsdviz
from upxo.interfaces.user_inputs.nbWidgets import selectProps_twinGS

_widgets = selectProps_twinGS()


In [ ]:
from upxo.interfaces.user_inputs.nbWidgets import readProps_twinGS

w = readProps_twinGS(_widgets)

from upxo.viz.ebsdviz import plot_grain_role_property_stats
fig, axes = plot_grain_role_property_stats(
    lfi         = rg.lfi_ebsd,
    parent_info = parent_info,
    prop        = rg.prop_ebsd,
    neigh_gid   = rg.neigh_gid_ebsd,
    selected_props  = w['selected_props'],
    selected_groups = w['selected_groups'],
    step_size   = rdr.step_size,
    bins        = 40,
    bw_method   = 'scott',
    peak_prominence = 0.02,
    figsize_per = (6, 4),
    dpi         = 100,
    suptitle    = 'Morphological & topological statistics by grain role — EBSD target',
    ncols       = w['ncols'],
    fontsize    = w['fontsize'],
)


## 9. Generate simulated grain structures


In [ ]:
import upxo.gsContainters.mcgs2Cont as gs_cntnr2_mc
MC_GS_Container2d = gs_cntnr2_mc.MC_GS_Container2d

MC_TIME_START, MC_TIME_END, MC_TIME_STEP = 2, -1, 2

cntr = MC_GS_Container2d.by_upxoMCSIM_gsset_GEN(indb = INPUT_DASHBOARD,
    mctimeStart = MC_TIME_START, mctimeStep = MC_TIME_STEP, mctimeEnd = MC_TIME_END,)

fig, axes = cntr.plot_temporal_distributions(
    props = ["area", "aspect_ratio", "solidity", "major_axis_length", "minor_axis_length"],
    ncols = 2, cmap = "plasma", fontsize = 12.0,
    suptitle = "Grain-property evolution across MC time slices",)
plt.show()

# Use cntr.stats_table to see the propertires.

In [ ]:
# GS SIMULATION
pxt = mcgs(input_dashboard=INPUT_DASHBOARD)
pxt.simulate(verbose=False)
pxt.detect_grains()
print(f'Temporal slices available: {pxt.m}')

mctimeStart = 2
mctimeStep = 1
mctimeEnd = -1

# --- characterise every temporal slice starting from 2nd ---
for tslice in pxt.m[mctimeStart:mctimeEnd:mctimeStep]:
    gs = pxt.gs[tslice]
    gs.char_morph_2d(
        use_version=2,
        bbox=True, bbox_ex=False,
        npixels=False, identify_pixel_locations=True, npixels_gb=False,
        area=True, aspect_ratio=True, solidity=True,
        major_axis_length=True, minor_axis_length=True,
        circularity=False, eccentricity=False, euler_number=False,
        moments_hu=False,
        append=False, saa=True, throw=False,
        char_grain_positions=False, find_neigh=False,
        char_gb=False, make_skim_prop=True,
        get_grain_coords=True,
    )
    print(f'  tslice={tslice:4d}  grains={gs.n}  '
          f'prop keys: {list(gs.prop.keys()) if hasattr(gs.prop, "keys") else type(gs.prop)}')


In [ ]:
# keep a reference to the last slice for downstream cells
tslice   = pxt.m[-1]
gstslice = pxt.gs[tslice]
print(f'\nActive slice set to tslice={tslice}  (last)')


MC simulation using `tgs2d.xls` dashboard. Every temporal slice (from index 2 onward) is characterised with morphology properties. `gstslice` holds the last slice.


In [ ]:
# All slices already characterised above.
# Inspect the prop DataFrame of the last (active) slice.
print(f'gstslice (tslice={tslice}) — prop shape: {gstslice.prop.shape}')
gstslice.prop.head()


Quick check: `prop` DataFrame shape and first rows for the active (last) temporal slice.


In [ ]:
print(f'Number of grains in experimental (EBSD) grain structure: {rdr.lfi_ebsd.max()}')

print('\nNumber of grains per characterised tslice (simulated):')
for ts in pxt.m:
    print(f'  tslice={ts:4d}  n_grains={pxt.gs[ts].n}')


Compares grain count of the EBSD target vs. each simulated slice — helps choose the best-matching temporal slice.


In [ ]:
## Select closest temporal slice for sim gs
tslice = pxt.m[TSLICE_INDEX]
gs = pxt.gs[tslice]
plt.imshow(gs.lfi)


## end


In [ ]:
import upxo.xtalphy.crystal_orientation as _co
import upxo.viz.ebsdviz as ebsdviz
from upxo.xtalphy.crystal_orientation import compute_grain_role_ratios
from upxo.viz.ebsdviz import print_grain_role_ratios

ratios = compute_grain_role_ratios(rg.lfi_ebsd, parent_info)
print_grain_role_ratios(ratios)


In [ ]:
## Twin thickness statistics of experimental (EBSD) grain structure
import upxo.xtalphy.crystal_orientation as _co
import upxo.viz.ebsdviz as ebsdviz
from upxo.xtalphy.crystal_orientation import compute_twin_thickness_stats
from upxo.viz.ebsdviz import plot_twin_thickness_stats

twin_thickness_stats = compute_twin_thickness_stats(
    parent_info  = parent_info,
    prop_ebsd    = rg.prop_ebsd,
    step_size_um = rdr.step_size,
)

print(f"Twin thickness statistics  [{twin_thickness_stats['col']}, µm]")
print(f"  Step size  : {twin_thickness_stats['step_um']} µm/px")
print(f"  N grains   : {len(twin_thickness_stats['thick_um'])}")
print(f"  Mean       : {twin_thickness_stats['mean']:.3f} µm")
print(f"  Median     : {twin_thickness_stats['median']:.3f} µm")
print(f"  Std dev    : {twin_thickness_stats['std']:.3f} µm")
print(f"  Min        : {twin_thickness_stats['min']:.3f} µm")
print(f"  Max        : {twin_thickness_stats['max']:.3f} µm")
print(f"  10th pct   : {twin_thickness_stats['pct10']:.3f} µm")
print(f"  90th pct   : {twin_thickness_stats['pct90']:.3f} µm")

fig, ax = plot_twin_thickness_stats(twin_thickness_stats)
plt.show()


## 15. Assign grain roles to simulated structure `gs`

Grain roles (pure parent / pure twin / intermediate / non-role) are assigned to the simulated grain structure by matching the **role fractions** measured from the EBSD target.  
Grains are sorted by area (largest → pure parent, next tier → pure twin, etc.), reflecting the physical expectation that parent grains are larger than their twins.


In [ ]:
import upxo.xtalphy.crystal_orientation as _co
import upxo.viz.ebsdviz as ebsdviz
from upxo.xtalphy.crystal_orientation import assign_grain_roles_by_ratio
from upxo.viz.ebsdviz import print_grain_role_ratios, plot_grain_role_map

# Build {grain_id: area} from gs.prop DataFrame
gs_grain_areas = dict(zip(gs.prop.index, gs.prop['area']))

# Assign roles to match EBSD target fractions (stored in `ratios`)
gs_roles = assign_grain_roles_by_ratio(gs.lfi, gs_grain_areas, ratios)

print('── Simulated gs — assigned role fractions ──')
print_grain_role_ratios(gs_roles)
print()
print('── EBSD target — reference fractions ──')
print_grain_role_ratios(ratios)


In [ ]:
fig, ax = plot_grain_role_map(
    gs.lfi,
    gs_roles,
    figsize=(10, 5),
    dpi=130,
    alpha_bg=0.08,
    suptitle='Simulated gs — grain roles assigned to match EBSD target fractions',
)


## Section 16 — CSL-based twin introduction in the simulated grain structure

Use the controls below to select a CSL boundary type and set twin geometry parameters.  
The pure-parent grains assigned in Section 15 will receive straight twin lamellae
generated by `Sline2d.by_LFAL` + `find_neigh_point_by_perp_distance`.


In [ ]:
# ── Twin-introduction parameter widgets ─────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display

_csl_options = list(selected_peaks.keys()) if selected_peaks else ['Σ3 (twin)']

_twin_widgets = {
    'csl_type': widgets.Dropdown(
        options=_csl_options,
        value=_csl_options[0],
        description='CSL type:',
        style={'description_width': '130px'},
    ),
    'twin_half_width': widgets.FloatSlider(
        value=2.0, min=1.0, max=10.0, step=0.5,
        description='Half-width (px):',
        style={'description_width': '130px'},
        layout=widgets.Layout(width='420px'),
    ),
    'width_perturb': widgets.FloatSlider(
        value=0.0, min=0.0, max=5.0, step=0.25,
        description='Width perturb (px):',
        style={'description_width': '130px'},
        layout=widgets.Layout(width='420px'),
    ),
    'n_twins': widgets.IntSlider(
        value=1, min=1, max=5, step=1,
        description='Twins / parent:',
        style={'description_width': '130px'},
        layout=widgets.Layout(width='420px'),
    ),
    'fixed_angle': widgets.FloatSlider(
        value=60.0, min=0.0, max=179.9, step=0.5,
        description='Twin angle (°):',
        style={'description_width': '130px'},
        layout=widgets.Layout(width='420px'),
    ),
    'angle_perturb': widgets.FloatSlider(
        value=0.0, min=0.0, max=15.0, step=0.5,
        description='Angle perturb (°):',
        style={'description_width': '130px'},
        layout=widgets.Layout(width='420px'),
    ),
    'random_angle': widgets.Checkbox(
        value=False,
        description='Random base angle per parent',
        indent=False,
    ),
    'rng_seed': widgets.IntText(
        value=42,
        description='RNG seed:',
        style={'description_width': '90px'},
        layout=widgets.Layout(width='220px'),
    ),
}

display(widgets.VBox([
    _twin_widgets['csl_type'],
    widgets.HTML('<b>Geometry</b>'),
    _twin_widgets['twin_half_width'],
    _twin_widgets['width_perturb'],
    _twin_widgets['n_twins'],
    widgets.HTML('<b>Orientation</b>'),
    _twin_widgets['fixed_angle'],
    _twin_widgets['angle_perturb'],
    _twin_widgets['random_angle'],
    widgets.HTML('<b>RNG</b>'),
    _twin_widgets['rng_seed'],
]))
print("Adjust parameters, then run the next cell.")


In [ ]:

# ── Introduce twins ──────────────────────────────────────────────────────────
import numpy as np
import upxo.geoEntities.sline2d as _sl_mod
import upxo.xtalphy.crystal_orientation as _co_mod
from upxo.xtalphy.crystal_orientation import introduce_twins_by_csl

# Read widget values
_csl_sel        = _twin_widgets['csl_type'].value
_half_width     = float(_twin_widgets['twin_half_width'].value)
_width_perturb  = float(_twin_widgets['width_perturb'].value)
_n_twins        = int(_twin_widgets['n_twins'].value)
_random_angle   = bool(_twin_widgets['random_angle'].value)
_angle          = None if _random_angle else float(_twin_widgets['fixed_angle'].value)
_angle_perturb  = float(_twin_widgets['angle_perturb'].value)
_seed           = int(_twin_widgets['rng_seed'].value)

# ── Match experimental twinned-grain percentage ──────────────────────────────
# In the EBSD, ratios['pure_parents']['ratio'] is the fraction of grains
# that host a twin.  Apply the same fraction to the simulated structure.
ebsd_twin_fraction = ratios['pure_parents']['ratio']
n_sim_total        = gs.n
n_to_twin          = max(1, round(ebsd_twin_fraction * n_sim_total))

all_parents_sorted = sorted(gs_roles['pure_parents']['grain_ids'])
rng_sub            = np.random.default_rng(_seed)
parent_gids        = set(
    rng_sub.choice(all_parents_sorted,
                   size=min(n_to_twin, len(all_parents_sorted)),
                   replace=False).tolist()
)

print(f"EBSD pure-parent fraction  : {ebsd_twin_fraction:.3f}  "
      f"({ratios['pure_parents']['count']} / {ratios['total']['count']} grains)")
print(f"Sim total grains           : {n_sim_total}")
print(f"Sim grains to twin         : {len(parent_gids)} "
      f"({len(parent_gids)/n_sim_total:.3f}  ≈ target {ebsd_twin_fraction:.3f})")
print()

# Work on a copy of the label field so gs.lfi is preserved
lfi_before = gs.lfi.copy()
lfi_twin   = lfi_before.copy()

print(f"Introducing twins into {len(parent_gids)} parent grain(s).")
print(f"  CSL type       : {_csl_sel}")
print(f"  Half-width     : {_half_width} ± {_width_perturb} px")
print(f"  Twins/parent   : {_n_twins}")
print(f"  Angle          : {'random' if _random_angle else f'{_angle}°'} ± {_angle_perturb}°")
print(f"  RNG seed       : {_seed}")

twin_result = introduce_twins_by_csl(
    lfi=lfi_twin,
    parent_grain_ids=parent_gids,
    csl_label=_csl_sel,
    twin_half_width=_half_width,
    twin_angle_deg=_angle,
    n_twins_per_parent=_n_twins,
    angle_perturb_deg=_angle_perturb,
    width_perturb=_width_perturb,
    rng_seed=_seed,
)

total_new = sum(len(v) for v in twin_result['new_twin_gids'].values())
print(f"\nDone. {total_new} new twin grain(s) created.")
print(f"Label field: {lfi_before.max()} → {twin_result['lfi'].max()} (max grain ID)")


In [ ]:
# ── Visualise before / after ─────────────────────────────────────────────────
import upxo.viz.ebsdviz as _ev_mod
from upxo.viz.ebsdviz import plot_twin_introduction_map

fig, axes = plot_twin_introduction_map(
    lfi_before=lfi_before,
    twin_result=twin_result,
    figsize=(14, 5),
    dpi=130,
    color_parent='#4878CF',
    color_twin='#D65F5F',
    color_other='#CCCCCC',
    alpha_other=0.30,
    show_cut_lines=True,
    suptitle=f'CSL twin introduction — {_csl_sel}  |  half-width={_half_width} px',
)


In [ ]:
# ── Verify that lfi_twin actually differs from lfi_before ───────────────────
import numpy as np

lfi_after = twin_result['lfi']

changed_mask  = lfi_before != lfi_after
n_changed     = changed_mask.sum()
n_total       = lfi_before.size
pct_changed   = 100.0 * n_changed / n_total

new_ids_in_after = set(np.unique(lfi_after)) - set(np.unique(lfi_before))

print(f"Pixels changed  : {n_changed:,} / {n_total:,}  ({pct_changed:.2f} %)")
print(f"New grain IDs   : {len(new_ids_in_after)}  (IDs {min(new_ids_in_after) if new_ids_in_after else 'n/a'}"
      f" – {max(new_ids_in_after) if new_ids_in_after else 'n/a'})")
print(f"lfi_before max  : {lfi_before.max()}")
print(f"lfi_after  max  : {lfi_after.max()}")
print(f"Arrays are identical: {np.array_equal(lfi_before, lfi_after)}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8), dpi=200)
im = ax.imshow(lfi_after, interpolation='nearest', aspect='equal', cmap='nipy_spectral')
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='Grain ID')
ax.set_title('lfi_twin — grain label field after twin introduction', fontsize=10)
ax.set_xlabel('x (pixels)')
ax.set_ylabel('y (pixels)')
plt.tight_layout()
plt.show()


In [ ]:
# ── Plot twinned phase only ──────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

lfi_after        = twin_result['lfi']
new_ids_in_after = sorted(set(np.unique(lfi_after)) - set(np.unique(lfi_before)))

# Pixels belonging to new twin grains; all others → NaN (transparent)
twin_only = np.where(np.isin(lfi_after, new_ids_in_after),
                     lfi_after.astype(float), np.nan)

fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
ax.imshow(np.ones_like(lfi_after), cmap='gray', vmin=0, vmax=1,
          interpolation='nearest', aspect='equal')          # white background
im = ax.imshow(twin_only, cmap='nipy_spectral',
               interpolation='nearest', aspect='equal',
               vmin=lfi_before.max() + 1, vmax=lfi_after.max())
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='Twin grain ID')
ax.set_title(f'Twinned phase only — {len(new_ids_in_after)} new twin grain(s)', fontsize=10)
ax.set_xlabel('x (pixels)')
ax.set_ylabel('y (pixels)')
plt.tight_layout()
plt.show()

# print(f'New twin grain IDs : {new_ids_in_after}')
print(f'Twin pixels        : {int(np.isfinite(twin_only).sum())}  '
      f'({100.0 * np.isfinite(twin_only).sum() / lfi_after.size:.2f} % of map)')


## Section 17 — Orientation assignment to match EBSD MDF

Assign crystallographic orientations (quaternions) to every domain in the
twinned simulated grain structure (`lfi_after`) so that the grain-boundary
Misorientation Distribution Function (MDF) of the simulation matches the
EBSD target.

**Strategy**
| Grain role | Orientation source |
|---|---|
| Pure-parent sim grains | Sampled from EBSD pure-parent pool |
| Introduced (geometric) twin grains | `q_twin = Σ3_Q ⊗ q_parent` (60°/⟨111⟩) |
| Pre-labelled pure-twin sim grains | Sampled from EBSD pure-twin pool |
| Intermediate sim grains | Sampled from EBSD intermediate pool |
| Non-role sim grains | Sampled from EBSD non-role pool |


In [ ]:
import upxo.xtalphy.crystal_orientation as _co
from upxo.xtalphy.crystal_orientation import assign_orientations_mdf_matched

ori_result = assign_orientations_mdf_matched(
    lfi_after      = lfi_after,
    gs_roles       = gs_roles,
    twin_result    = twin_result,
    ebsd_lfi       = rg.lfi_ebsd,
    ebsd_quat      = rg.quat_ebsd,
    ebsd_parent_info = parent_info,
    rng_seed       = 42,
)
print('Assigned orientations to', len(ori_result['grain_quats']), 'sim grains')
print('quat_pixel shape :', ori_result['quat_pixel'].shape)
print('Neighbour-graph entries:', len(ori_result['neigh_gid_sim']))


In [ ]:
from upxo.xtalphy.crystal_orientation import compute_mdf_from_quats
import matplotlib.pyplot as plt
import numpy as np

# MDF of simulated structure
mdf_sim = compute_mdf_from_quats(
    lfi       = lfi_after,
    quat      = ori_result['quat_pixel'],
    neigh_gid = ori_result['neigh_gid_sim'],
)

# Overlay comparison: EBSD vs sim MDF
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(mdf['hist_bin_centers'],     mdf['hist_density'],     'k-',  lw=2, label='EBSD')
ax.plot(mdf_sim['hist_bin_centers'], mdf_sim['hist_density'], 'r--', lw=2, label='Simulated (Section 17)')
ax.set_xlabel('Disorientation angle (°)', fontsize=12)
ax.set_ylabel('Probability density',      fontsize=12)
ax.set_title('MDF comparison: EBSD vs simulated structure')
ax.legend()
fig.tight_layout()
plt.show()

print(f"EBSD   mean disorientation: {mdf['mean_angle']:.2f}°  ±{mdf['std_angle']:.2f}°")
print(f"Sim    mean disorientation: {mdf_sim['mean_angle']:.2f}°  ±{mdf_sim['std_angle']:.2f}°")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualise the w-component of the per-pixel quaternion map as a proxy
# orientation map.  w = cos(θ/2), so this image encodes the half-angle
# of the rotation and gives a visually distinct colour per orientation family.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Simulated structure: w-channel
ax = axes[0]
im0 = ax.imshow(ori_result['quat_pixel'][..., 0], cmap='hsv', origin='upper',
                vmin=0, vmax=1)
ax.set_title('Sim structure – quaternion w-channel\n(proxy orientation map)', fontsize=11)
ax.axis('off')
plt.colorbar(im0, ax=ax, fraction=0.046, label='w = cos(θ/2)')

# Show label field with twin outlines for reference
ax2 = axes[1]
ax2.imshow(lfi_after, cmap='tab20b', origin='upper', interpolation='nearest')
ax2.set_title('Simulated grain labels (lfi_after)', fontsize=11)
ax2.axis('off')

fig.suptitle('Section 17 — Orientation map and grain labels', fontsize=12)
fig.tight_layout()
plt.show()

# --- Summary statistics ---
n_intro_twins = sum(len(v) for v in twin_result.get('new_twin_gids', {}).values())
print(f"Total sim grains with assigned orientation : {len(ori_result['grain_quats'])}")
print(f"  of which geometrically-introduced twins  : {n_intro_twins}")
print(f"  CSL label used for twin introduction     : {twin_result.get('csl_label', 'N/A')}")


In [ ]:
## Scale factor: match synthetic mean grain size to EBSD mean grain size
import upxo.xtalphy.crystal_orientation as _co
import upxo.viz.ebsdviz as ebsdviz
from upxo.xtalphy.crystal_orientation import compute_scale_factor_grain_size
from upxo.viz.ebsdviz import plot_grain_area_comparison

scale_result = compute_scale_factor_grain_size(
    lfi_sim            = lfi_after,
    mean_area_ebsd_um2 = rg.stat_ebsd['area']['mean'],
    ebsd_step_size_um  = rdr.step_size,
    prop_ebsd          = rg.prop_ebsd,
)

print("─── Grain-size matching scale factor ───────────────────────────────")
print(f"  EBSD mean grain area         : {scale_result['mean_area_ebsd_um2']:.3f} µm²"
      f"  (step = {scale_result['ebsd_step_size_um']} µm/px,"
      f"  N = {rg.stat_ebsd['area']['count']} grains)")
print(f"  Synthetic mean grain area    : {scale_result['mean_area_sim_px2']:.1f} px²"
      f"  (N = {len(scale_result['gids_sim'])} grains in lfi_after)")
print()
print(f"  Scale factor                 : {scale_result['scale_factor']:.5f}  µm / sim-pixel")
print(f"  Sim pixel size (scaled)      : {scale_result['scale_factor']:.5f} µm")
print(f"  EBSD step size               : {scale_result['ebsd_step_size_um']:.4f} µm")
print(f"  Ratio (sim/EBSD pixel size)  : {scale_result['ratio_pixel_sizes']:.4f}")
print()
print(f"  Synthetic domain size        : {scale_result['sim_domain_px'][1]} × {scale_result['sim_domain_px'][0]} px")
print(f"  Scaled physical domain size  : {scale_result['sim_domain_um'][1]:.1f} × {scale_result['sim_domain_um'][0]:.1f} µm")
print()
match = abs(scale_result['mean_area_sim_um2'] - scale_result['mean_area_ebsd_um2']) < 1e-6
print(f"  Sim mean area after scaling  : {scale_result['mean_area_sim_um2']:.3f} µm²"
      f"  (target: {scale_result['mean_area_ebsd_um2']:.3f} µm²)"
      + ("  ✓" if match else ""))

fig, ax = plot_grain_area_comparison(scale_result)
plt.show()


# Generating smoothed grain structure


In [ ]:
# ── Smoothing pipeline ────────────────────────────────────────────────────────
lfi = lfi_after.copy()  # work on a copy to preserve original

neigh_gid = find_neighs2d(lfi, conn=4)
areas = np.bincount(lfi.ravel())[1:]

def merge_small_grains(neigh_gid, areas, lfi, athreshold=1):
    """
    Merge small grains with their dominant (largest) neighboring grain.
    Recomputes connected components and updates neighbor/area data after each merger.
    """
    def find_dominant_neighbor(gid, neigh_gid, areas):
        """Find the dominant (largest area) neighbor for a grain."""
        if gid in neigh_gid and len(neigh_gid[gid]) > 0:
            neighbors = neigh_gid[gid]
            neighbor_areas = areas[neighbors - 1]
            return neighbors[np.argmax(neighbor_areas)]
        return gid
    lfi_merged = deepcopy(lfi)
    # Identify and merge small grains
    small_grains = np.where(areas <= athreshold)[0] + 1
    for gid in small_grains:
        dominant = find_dominant_neighbor(gid, neigh_gid, areas)
        lfi_merged[lfi_merged == gid] = dominant
        lfi_merged = cc3d.connected_components(lfi_merged, connectivity=4)
        # Update neighbor and area data after merger
        neigh_gid = find_neighs2d(lfi_merged, conn=4)
        areas = np.bincount(lfi_merged.ravel())[1:]
    return lfi_merged

lfi_merged = merge_small_grains(neigh_gid, areas, lfi, athreshold=AREA_THRESHOLD)

seeds = gridOps.generate_constrained_hybrid_seeds(
    lfi, target_spacing=1.0, bulk_spacing=1.0, jitter_factor=0.25,
    margin=0.5, padding=2.0, plot_seeds=True, figsize=(5, 5), dpi=150, markersize=0.5
)

manifold_structure = GrainManifold2D.by_tessellation(lfi, seeds)
original_cells = {gid: geom for gid, geom in manifold_structure.cells.items()}

gsviz.plot_manifold_geom([original_cells], figsize=(12, 6), dpi=100, inlude_legend=False)

manifold_structure.smooth_interfaces(iterations=SMOOTH_ITER, lmbda=SMOOTH_LAMBDA, mu=SMOOTH_MU)

gsviz.plot_manifold_geom(
    [original_cells, manifold_structure.cells], figsize=(12, 6), dpi=100, inlude_legend=False
)

_ = manifold_structure.trim_to_rve(bounds=TRIM_BOUNDS)

gsviz.plot_manifold_geom(
    [original_cells, manifold_structure.cells], figsize=(12, 6), dpi=200, inlude_legend=False
)

# Remove empty/invalid polygons
cleaned_cells = {}
for gid, geom in manifold_structure.cells.items():
    if geom is not None and geom.is_valid and geom.area > 0:
        cleaned_cells[gid] = geom
manifold_structure.cells = cleaned_cells

print(f"Removed {len(manifold_structure.cells) - len(cleaned_cells)} empty/invalid polygons")
print(f"Remaining polygons: {len(cleaned_cells)}")

gsviz.plot_manifold_geom([manifold_structure.cells], figsize=(12, 6), dpi=150, inlude_legend=False)


In [ ]:
from collections import Counter
"""Construct neighbor dictionary for all polygons with integer GIDs starting from 1"""
polygon_neighbors = {}
"""Get list of GIDs and ensure they're integers in sorted order"""
gids = sorted([int(gid) for gid in manifold_structure.cells.keys()])
"""For each polygon, find all neighbors (polygons that touch it)"""
for gid1 in gids:
    polygon_neighbors[gid1] = []
    geom1 = manifold_structure.cells[gid1]
    """Compare with all other polygons"""
    for gid2 in gids:
        if gid1 != gid2:
            geom2 = manifold_structure.cells[gid2]
            """Check if polygons touch (share boundary but don't overlap)"""
            if geom1.touches(geom2):
                polygon_neighbors[gid1].append(gid2)
# Summary
print(f"Total polygons: {len(polygon_neighbors)}")
print(f"\nNeighbor statistics:")
neighbor_counts = [len(neighs) for neighs in polygon_neighbors.values()]
print(f"  Min neighbors: {min(neighbor_counts)}")
print(f"  Max neighbors: {max(neighbor_counts)}")
print(f"  Avg neighbors: {sum(neighbor_counts) / len(neighbor_counts):.2f}")
# grain_ids, neigh_id = find_neighs()
neigh_dist = Counter(neighbor_counts)
print(f"\nNeighbor count distribution:")
for n_neighs in sorted(neigh_dist.keys()):
    count = neigh_dist[n_neighs]
    print(f"  {n_neighs} neighbors: {count} polygons")

print(f"\nSample neighbors:")
for gid in list(polygon_neighbors.keys())[:5]:
    print(f"  GID {gid}: {polygon_neighbors[gid]}")


In [ ]:
"""manifold_structure.cells key renumbering to start from 1 contiguously"""
old_to_new_gid = {}
new_cells = {}
"""Debug: Check GIDs in both dictionaries"""
cells_gids = set(manifold_structure.cells.keys())
neighbors_gids = set(polygon_neighbors.keys())
all_neighbor_refs = set()
for neighbors in polygon_neighbors.values():
    all_neighbor_refs.update(neighbors)

print(f"Debug - GIDs in manifold_structure.cells: {len(cells_gids)}")
print(f"Debug - GIDs in polygon_neighbors keys: {len(neighbors_gids)}")
print(f"Debug - All neighbor GID references: {len(all_neighbor_refs)}")

"""Check for mismatches"""
missing_in_cells = all_neighbor_refs - cells_gids
missing_in_neighbors = cells_gids - neighbors_gids
if missing_in_cells:
    print(f"Warning: Neighbor references point to missing cells: {missing_in_cells}")
    print(f"Filtering polygon_neighbors to only include cells in manifold_structure.cells")
    # Filter polygon_neighbors to only include cells that exist
    polygon_neighbors = {gid: neighbors for gid, neighbors in polygon_neighbors.items() if gid in cells_gids}
    
if missing_in_neighbors:
    print(f"Warning: Cells missing from polygon_neighbors: {missing_in_neighbors}")

for new_gid, old_gid in enumerate(sorted(manifold_structure.cells.keys()), start=1):
    old_to_new_gid[old_gid] = new_gid
    new_cells[new_gid] = manifold_structure.cells[old_gid]
manifold_structure.cells = new_cells

"""Update polygon_neighbors with new GID mapping"""
polygon_neighbors_renumbered = {}
for old_gid, neighbors in polygon_neighbors.items():
    new_gid = old_to_new_gid[old_gid]
    new_neighbors = [old_to_new_gid[n] for n in neighbors if n in old_to_new_gid]
    polygon_neighbors_renumbered[new_gid] = new_neighbors
polygon_neighbors = polygon_neighbors_renumbered

print(f"\nRenumbered {len(old_to_new_gid)} grain cells to contiguous IDs from 1 to {len(new_cells)}")
print(f"Sample mapping (old -> new): {dict(list(old_to_new_gid.items())[:5])}")


In [ ]:
"""Validation check for all polygons in manifold_structure.cells"""
from shapely.geometry import Polygon

validity_report = {}
for gid, geom in manifold_structure.cells.items():
    is_valid = geom.is_valid if geom is not None else False
    is_polygon = isinstance(geom, Polygon) if geom is not None else False
    has_area = geom.area > 0 if geom is not None else False
    validity_report[gid] = {'is_valid': is_valid, 'has_area': has_area, 'is_polygon': is_polygon, 'area': geom.area if geom is not None else 0}

"""Summary"""
print(f"Total polygons: {len(validity_report)}")
valid_count = sum(1 for v in validity_report.values() if v['is_valid'])
area_count = sum(1 for v in validity_report.values() if v['has_area'])
polygon_count = sum(1 for v in validity_report.values() if v['is_polygon'])

print(f"Valid polygons: {valid_count}")
print(f"Polygons with area: {area_count}")
print(f"Polygon type objects: {polygon_count}")

print(f"\nInvalid polygons:")
invalid_polys = {gid: v for gid, v in validity_report.items() if not v['is_valid']}
if invalid_polys:
    for gid, vals in invalid_polys.items():
        print(f"  GID {gid}: {vals}")
else:
    print("  None - all polygons are valid!")


In [ ]:
"""Ensure manifold_structure.cells has integer keys starting from 1"""
cells_int = {int(gid) if isinstance(gid, str) else gid: geom 
             for gid, geom in manifold_structure.cells.items()}
manifold_structure.cells = cells_int
"""Construct neighbor dictionary for all polygons with integer GIDs starting from 1"""
polygon_neighbors = {}
"""Get list of GIDs and ensure they're integers in sorted order"""
gids = sorted([int(gid) for gid in manifold_structure.cells.keys()])
"""For each polygon, find all neighbors (polygons that touch it)"""
for gid1 in gids:
    polygon_neighbors[gid1] = []
    geom1 = manifold_structure.cells[gid1]
    """Compare with all other polygons"""
    for gid2 in gids:
        if gid1 != gid2:
            geom2 = manifold_structure.cells[gid2]
            """Check if polygons touch (share boundary but don't overlap)"""
            if geom1.touches(geom2):
                polygon_neighbors[gid1].append(gid2)

print(f"Total polygons: {len(polygon_neighbors)}")
print(f"\nNeighbor statistics:")
neighbor_counts = [len(neighs) for neighs in polygon_neighbors.values()]
print(f"  Min neighbors: {min(neighbor_counts)}")
print(f"  Max neighbors: {max(neighbor_counts)}")
print(f"  Avg neighbors: {sum(neighbor_counts) / len(neighbor_counts):.2f}")

"""Show polygons by neighbor count"""
from collections import Counter
neigh_dist = Counter(neighbor_counts)
print(f"\nNeighbor count distribution:")
for n_neighs in sorted(neigh_dist.keys()):
    count = neigh_dist[n_neighs]
    print(f"  {n_neighs} neighbors: {count} polygons")

print(f"\nSample neighbors:")
for gid in list(polygon_neighbors.keys())[:5]:
    print(f"  GID {gid}: {polygon_neighbors[gid]}")


In [ ]:
polygon_neighbors


In [ ]:
"""Construct grain-to-grain interaction pairs (symmetric)"""
cell_pairs = set()
for gid1, neighbors in polygon_neighbors.items():
    for gid2 in neighbors:
        """Create symmetric pairs by always putting smaller ID first"""
        pair = tuple(sorted([gid1, gid2]))
        cell_pairs.add(pair)
"""Convert to sorted list for easier inspection"""
cell_pairs_list = sorted(list(cell_pairs))

print(f"Total cell interaction pairs: {len(cell_pairs_list)}")
print(f"\nFirst 20 pairs:")
for pair in cell_pairs_list[:20]:
    print(f"  {pair}")

print(f"\nSample statistics:")
print(f"  Min cell ID in pairs: {min(min(p) for p in cell_pairs_list)}")
print(f"  Max cell ID in pairs: {max(max(p) for p in cell_pairs_list)}")


In [ ]:
from shapely.geometry import MultiLineString

"""Create a dictionary mapping each cell pair to its shared interface (MultiLineString)"""
cell_pair_interfaces = {}

for pair in cell_pairs_list:
    gid1, gid2 = pair
    geom1 = manifold_structure.cells[gid1]
    geom2 = manifold_structure.cells[gid2]
    """Get the intersection of the two polygon boundaries.
    This gives us the shared interface as a LineString or MultiLineString"""
    interface = geom1.boundary.intersection(geom2.boundary)
    cell_pair_interfaces[pair] = interface

print(f"Total interfaces extracted: {len(cell_pair_interfaces)}")

"""Categorize interfaces by geometry type"""
interface_types = {}
print(f"\nSample interfaces (first 10 pairs):")
for i, (pair, interface) in enumerate(list(cell_pair_interfaces.items())[:10]):
    geom_type = interface.geom_type
    print(f"  Pair {pair}: {geom_type} (length: {interface.length:.3f})")
    interface_types[geom_type] = interface_types.get(geom_type, 0) + 1  # Count by type

"""Full count across all interfaces"""
for pair, interface in cell_pair_interfaces.items():
    geom_type = interface.geom_type
    interface_types[geom_type] = interface_types.get(geom_type, 0) + 1

print(f"\nInterface geometry type summary:")
for geom_type, count in sorted(interface_types.items()):
    print(f"  {geom_type}: {count}")


In [ ]:
from collections import defaultdict
from shapely.geometry import Point, Polygon, MultiPolygon

"""Tolerance for grouping nearly-identical coordinates after smoothing"""
coord_decimals = 6
vertex_to_gids = defaultdict(set)

for gid, geom in manifold_structure.cells.items():
    if geom is None or geom.is_empty:
        continue
    polys = [geom] if isinstance(geom, Polygon) else list(geom.geoms) if isinstance(geom, MultiPolygon) else []
    for poly in polys:
        # Exterior ring
        for x, y in list(poly.exterior.coords)[:-1]:  # skip duplicate closing vertex
            key = (round(float(x), coord_decimals), round(float(y), coord_decimals))
            vertex_to_gids[key].add(int(gid))
        # Interior rings (if any)
        for ring in poly.interiors:
            for x, y in list(ring.coords)[:-1]:
                key = (round(float(x), coord_decimals), round(float(y), coord_decimals))
                vertex_to_gids[key].add(int(gid))

"""Keep only true junctions (>=3 grains meeting)"""
junction_items = sorted([(coord, tuple(sorted(gids))) for coord, gids in vertex_to_gids.items() if len(gids) >= 3], key=lambda t: (t[0][0], t[0][1]) )

"""Required list of shapely Point objects"""
junction_points = [Point(x, y) for (x, y), _ in junction_items]

"""JP dictionary: key = JP ID (1-based), value = [order, participating grain IDs tuple]"""
jp_dict = {jp_id: [len(gids), gids] for jp_id, (_, gids) in enumerate(junction_items, start=1)}

print(f"Total junction points: {len(junction_points)}")
print("Sample jp_dict entries:", dict(list(jp_dict.items())[:5]))


# Section 19 — Conformal Finite Element Meshing with Gmsh

Generate a **conformal 2-D FE mesh** from the smoothed grain structure stored in `manifold_structure.cells`.

**Strategy**
1. Flatten all `MULTIPOLYGON` grains into individual `POLYGON` fragments.
2. Build the gmsh geometry from Shapely polygon rings — each exterior ring becomes a `Curve Loop` + `Plane Surface`.
3. Shared grain-boundary edges are handled automatically because adjacent polygons share coordinate-identical vertices (enforced by Shapely's `snap` / `touches` conformality).
4. Assign one **Physical Surface** per grain and one **Physical Curve** per grain-boundary pair.
5. Set mesh size and generate the triangular mesh.


## Step 19.1 — Flatten MultiPolygons and inspect grain geometry inventory


In [ ]:
from shapely.geometry import Polygon, MultiPolygon

# ── Flatten MultiPolygons → individual Polygon fragments ─────────────────────
flat_cells = {}   # flat_id -> Shapely Polygon
gid_map    = {}   # flat_id -> original grain ID

flat_id = 1
for gid, geom in manifold_structure.cells.items():
    if isinstance(geom, Polygon):
        parts = [geom]
    elif isinstance(geom, MultiPolygon):
        parts = list(geom.geoms)
    else:
        parts = []
    for part in parts:
        if part is not None and part.area > 0:
            flat_cells[flat_id] = part
            gid_map[flat_id]    = gid
            flat_id += 1

print(f"Original grains : {len(manifold_structure.cells)}")
print(f"Flat polygons   : {len(flat_cells)}")
print(f"(MultiPolygon parts expanded individually)")

## Step 19.2 — Mesh size parameters


In [ ]:
print(f"GB mesh size       : {MESH_SIZE_GB} px")
print(f"Bulk mesh size     : {MESH_SIZE_BULK} px")
print(f"Element order      : {MESH_ORDER}")
print(f"Algorithm          : {MESH_ALGO}")
print(f"Recombine to quads : {RECOMBINE_TO_QUADS}")


## Step 19.3 — Build gmsh geometry from Shapely polygons


In [ ]:
import gmsh

gmsh.initialize()
gmsh.model.add("grain_structure")
gmsh.option.setNumber("General.Terminal", 0)   # suppress gmsh console output

# ── Coordinate tolerance for point deduplication ────────────────────────────
COORD_TOL     = 1e-6
point_registry = {}   # (rx, ry) -> gmsh point tag

def get_or_add_point(x, y, lc):
    """Return existing gmsh point tag or create a new one."""
    key = (round(x, 6), round(y, 6))
    if key not in point_registry:
        tag = gmsh.model.geo.addPoint(x, y, 0.0, lc)
        point_registry[key] = tag
    return point_registry[key]

# ── Dictionaries to store gmsh entity tags ──────────────────────────────────
surface_tags = {}   # flat_id -> gmsh surface tag
loop_tags    = {}   # flat_id -> gmsh curve loop tag

for flat_id, poly in flat_cells.items():
    # ── Exterior ring ────────────────────────────────────────────────────────
    coords = list(poly.exterior.coords)[:-1]   # drop duplicate closing vertex
    pt_tags = [get_or_add_point(x, y, MESH_SIZE_GB) for x, y in coords]

    # Build lines between consecutive points (wrap-around)
    line_tags = []
    for i in range(len(pt_tags)):
        p1 = pt_tags[i]
        p2 = pt_tags[(i + 1) % len(pt_tags)]
        line_tags.append(gmsh.model.geo.addLine(p1, p2))

    outer_loop = gmsh.model.geo.addCurveLoop(line_tags)
    inner_loops = []

    # ── Interior rings (holes) ───────────────────────────────────────────────
    for ring in poly.interiors:
        rcoords  = list(ring.coords)[:-1]
        rpt_tags = [get_or_add_point(x, y, MESH_SIZE_GB) for x, y in rcoords]
        rlines   = []
        for i in range(len(rpt_tags)):
            p1 = rpt_tags[i]
            p2 = rpt_tags[(i + 1) % len(rpt_tags)]
            rlines.append(gmsh.model.geo.addLine(p1, p2))
        inner_loops.append(gmsh.model.geo.addCurveLoop(rlines))

    surf_tag = gmsh.model.geo.addPlaneSurface([outer_loop] + inner_loops)
    surface_tags[flat_id] = surf_tag
    loop_tags[flat_id]    = outer_loop

gmsh.model.geo.synchronize()

print(f"gmsh points created  : {len(point_registry)}")
print(f"gmsh surfaces created: {len(surface_tags)}")

## Step 19.4 — Assign Physical Groups (one per original grain)


In [ ]:
from collections import defaultdict

# Group flat_ids back to original grain IDs for Physical Surface assignment
orig_to_flat = defaultdict(list)
for flat_id, orig_gid in gid_map.items():
    orig_to_flat[orig_gid].append(surface_tags[flat_id])

physical_surface_tags = {}   # orig_gid -> gmsh physical group tag
for orig_gid, surfs in orig_to_flat.items():
    pg_tag = gmsh.model.addPhysicalGroup(2, surfs, tag=orig_gid)
    gmsh.model.setPhysicalName(2, pg_tag, f"grain_{orig_gid}")
    physical_surface_tags[orig_gid] = pg_tag

print(f"Physical Surface groups assigned: {len(physical_surface_tags)}")
print(f"Sample (orig_gid -> pg_tag): {dict(list(physical_surface_tags.items())[:6])}")

## Step 19.5 — Set mesh sizes and generate the 2D mesh


In [ ]:
import time

# ── Mesh algorithm and size field ────────────────────────────────────────────
gmsh.option.setNumber("Mesh.Algorithm",              MESH_ALGO)
gmsh.option.setNumber("Mesh.ElementOrder",           MESH_ORDER)
gmsh.option.setNumber("Mesh.CharacteristicLengthMin", MESH_SIZE_GB * 0.5)
gmsh.option.setNumber("Mesh.CharacteristicLengthMax", MESH_SIZE_BULK)
# Prevent boundary lc values from propagating into grain interiors;
# without this, all points added with lc=MESH_SIZE_GB flood the domain
# and MESH_SIZE_BULK has no effect.
gmsh.option.setNumber("Mesh.CharacteristicLengthExtendFromBoundary", 0)

# Distance-based size field: finer near grain boundary curves, coarser in bulk
field_dist = gmsh.model.mesh.field.add("Distance")
all_curves = [tag for _, tag in gmsh.model.getEntities(1)]
gmsh.model.mesh.field.setNumbers(field_dist, "CurvesList", all_curves)
gmsh.model.mesh.field.setNumber(field_dist,  "Sampling",   50)

field_thresh = gmsh.model.mesh.field.add("Threshold")
gmsh.model.mesh.field.setNumber(field_thresh, "InField",   field_dist)
gmsh.model.mesh.field.setNumber(field_thresh, "SizeMin",   MESH_SIZE_GB)
gmsh.model.mesh.field.setNumber(field_thresh, "SizeMax",   MESH_SIZE_BULK)
gmsh.model.mesh.field.setNumber(field_thresh, "DistMin",   0.5)
gmsh.model.mesh.field.setNumber(field_thresh, "DistMax",   5.0)

gmsh.model.mesh.field.setAsBackgroundMesh(field_thresh)

# ── Recombine triangles → quads (Alg 8 only) ─────────────────────────────────
if MESH_ALGO == 8 and RECOMBINE_TO_QUADS:
    for _surf in surface_tags.values():
        gmsh.model.mesh.setRecombine(2, _surf)

# ── Generate mesh ────────────────────────────────────────────────────────────
t0 = time.perf_counter()
gmsh.model.mesh.generate(2)
elapsed = time.perf_counter() - t0

# ── Statistics ───────────────────────────────────────────────────────────────
node_tags, _, _ = gmsh.model.mesh.getNodes()
elem_types, elem_tags, _ = gmsh.model.mesh.getElements(dim=2)

n_nodes = len(node_tags)
n_elems = sum(len(et) for et in elem_tags)

print(f"Mesh generated in {elapsed:.2f} s")
print(f"  Nodes    : {n_nodes:,}")
print(f"  Elements : {n_elems:,}")
print(f"  Types    : {elem_types}  (3=Tri3, 9=Tri6, 2=Tri3, 16=Quad4, 10=Quad8/9)")

## Step 19.6 — Visualise mesh with Matplotlib


In [ ]:
from upxo.viz.meshviz import see_femesh

# ── Extract nodes from gmsh ──────────────────────────────────────────────────
node_tags, node_coords, _ = gmsh.model.mesh.getNodes()
coords_2d = node_coords.reshape(-1, 3)[:, :2]   # x, y only

# node_tags are 1-based; build a fast lookup array
max_tag    = int(node_tags.max())
tag_to_idx = np.zeros(max_tag + 1, dtype=int)
tag_to_idx[node_tags.astype(int)] = np.arange(len(node_tags))

# ── Collect triangles (Tri3=2, Tri6=9) and quads (Quad4=3, Quad8=16, Quad9=10)
tri_verts  = []
quad_verts = []

for etype, etags, enodes in zip(*gmsh.model.mesh.getElements(dim=2)):
    n_per_elem = len(enodes) // len(etags)
    corners    = enodes.reshape(-1, n_per_elem)
    idx        = tag_to_idx[corners.astype(int)]
    if etype in (2, 9):         # Tri3 or Tri6 — keep first 3 corner nodes
        tri_verts.append(idx[:, :3])
    elif etype in (3, 16, 10):  # Quad4, Quad8, Quad9 — keep first 4 corner nodes
        quad_verts.append(idx[:, :4])

tri_verts  = np.vstack(tri_verts)  if tri_verts  else np.empty((0, 3), dtype=int)
quad_verts = np.vstack(quad_verts) if quad_verts else np.empty((0, 4), dtype=int)

# ── Visualise via see_femesh ─────────────────────────────────────────────────
fig, ax = see_femesh(
    points_2d      = coords_2d,
    lines          = None,
    triangles      = tri_verts  if len(tri_verts)  else None,
    quads          = quad_verts if len(quad_verts) else None,
    figsize        = (12, 9),
    dpi            = 150,
    tri_edgecolor  = 'steelblue',
    tri_linewidth  = 0.2,
    tri_alpha      = 0.7,
    quad_edgecolor = 'darkorange',
    quad_linewidth = 0.2,
    quad_alpha     = 0.7,
    show_axis      = True,
    show_stats     = True,
    title          = (f'Conformal FE mesh — {len(tri_verts):,} tri + '
                      f'{len(quad_verts):,} quad elements, {len(coords_2d):,} nodes'),
)

# Overlay grain boundary polygons (Shapely exterior rings)
for poly in flat_cells.values():
    xs, ys = poly.exterior.xy
    ax.plot(xs, ys, 'k-', linewidth=0.6, zorder=2)

plt.tight_layout()
plt.show()

print(f"Triangles plotted : {len(tri_verts):,}")
print(f"Quads plotted     : {len(quad_verts):,}")
print(f"Nodes plotted     : {len(coords_2d):,}")

## Step 19.7 — Export mesh

Export to `.msh` (gmsh native), `.inp` (Abaqus), or `.vtk`.  
Set the desired path and format below.


In [ ]:
# Step 19.7 — export handled inside mesh_gs() via femesh_gmsh.
# Files were written before gmsh.finalize() was called internally.
if mesh_result['exported']:
    print('Exported files:')
    for p in mesh_result['exported']:
        print(f'  {p}')
else:
    print('No files exported (set out_dir and formats in mesh_gs() to export).')

In [ ]:
manifold_structure.cells
